# MedFusionNet — Colab Training, Resume, Test And Grad-CAM

Ce notebook couvre le workflow complet :
- télécharger le dataset si le dossier `data/` n'est pas encore sur Drive,
- lancer un **test rapide** d'1 époque,
- lancer ensuite un **entraînement sérieux**,
- tester le meilleur checkpoint,
- visualiser un **Grad-CAM**.

Points importants :
- `venv` et `__pycache__` ne sont pas nécessaires sur Drive,
- les checkpoints et métriques sont sauvegardés sur Drive,
- si vous gardez le même `RUN_NAME`, le notebook reprend automatiquement depuis `last.pth`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import shutil
import subprocess
import sys

PROJECT_DRIVE_DIR = Path('/content/drive/MyDrive/DPR_PFA4IADO')
RUN_NAME = 'smoke_colab_v1'
COPY_DATA_TO_RUNTIME = True
DOWNLOAD_DATA_IF_MISSING = True
KAGGLE_DATASET = 'paultimothymooney/chest-xray-pneumonia'
KAGGLE_JSON_DRIVE_PATH = Path('/content/drive/MyDrive/kaggle.json')

PROJECT_SRC = PROJECT_DRIVE_DIR / 'DPR_MedFusionNet'
DATA_DRIVE_DIR = PROJECT_SRC / 'data'
DOWNLOAD_CACHE_DIR = PROJECT_DRIVE_DIR / '_kaggle_cache'
RUNTIME_ROOT = Path('/content/medfusionnet_colab')
RUNTIME_DIR = RUNTIME_ROOT / 'DPR_MedFusionNet'
RUNS_ROOT = PROJECT_DRIVE_DIR / 'colab_runs'

assert PROJECT_SRC.exists(), f'Projet introuvable: {PROJECT_SRC}'
print('PROJECT_SRC            =', PROJECT_SRC)
print('DATA_DRIVE_DIR         =', DATA_DRIVE_DIR)
print('KAGGLE_JSON_DRIVE_PATH =', KAGGLE_JSON_DRIVE_PATH)
print('RUNS_ROOT              =', RUNS_ROOT)


In [ ]:
# Installation légère pour Colab.
req_file = PROJECT_SRC / 'requirements-colab.txt'
if not req_file.exists():
    req_file = PROJECT_SRC / 'requirements.txt'

print('Installing from:', req_file)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(req_file)], check=True)

import torch
print('Torch      :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())
print('Device     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')


In [ ]:
# Télécharge le dataset Kaggle sur Drive si le dossier data/ n'existe pas encore.
def dataset_ready(root: Path) -> bool:
    required = [root / 'train', root / 'val', root / 'test']
    return all(path.exists() for path in required) and any((root / 'train').rglob('*'))

if dataset_ready(DATA_DRIVE_DIR):
    print('Dataset déjà présent sur Drive:', DATA_DRIVE_DIR)
else:
    assert DOWNLOAD_DATA_IF_MISSING, 'Le dataset est absent et DOWNLOAD_DATA_IF_MISSING=False.'
    assert KAGGLE_JSON_DRIVE_PATH.exists(), (
        f'Clé Kaggle introuvable: {KAGGLE_JSON_DRIVE_PATH}\n'
        'Téléchargez votre fichier kaggle.json depuis Kaggle > Account > API, puis placez-le sur Drive.'
    )

    kaggle_dir = Path.home() / '.kaggle'
    kaggle_dir.mkdir(parents=True, exist_ok=True)
    kaggle_json_runtime = kaggle_dir / 'kaggle.json'
    shutil.copy2(KAGGLE_JSON_DRIVE_PATH, kaggle_json_runtime)
    os.chmod(kaggle_json_runtime, 0o600)

    DOWNLOAD_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    zip_path = DOWNLOAD_CACHE_DIR / 'chest-xray-pneumonia.zip'
    extract_dir = DOWNLOAD_CACHE_DIR / 'extracted'
    source_root = extract_dir / 'chest_xray'

    if not zip_path.exists():
        print('Téléchargement du dataset Kaggle...')
        subprocess.run([
            'kaggle', 'datasets', 'download',
            '-d', KAGGLE_DATASET,
            '-p', str(DOWNLOAD_CACHE_DIR),
        ], check=True)
    else:
        print('Archive Kaggle déjà présente :', zip_path)

    if not source_root.exists():
        if extract_dir.exists():
            shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        print('Extraction du dataset...')
        shutil.unpack_archive(str(zip_path), str(extract_dir))
    else:
        print('Dataset déjà extrait :', source_root)

    assert source_root.exists(), f'Dossier source introuvable après extraction: {source_root}'

    if DATA_DRIVE_DIR.exists():
        shutil.rmtree(DATA_DRIVE_DIR)
    DATA_DRIVE_DIR.mkdir(parents=True, exist_ok=True)

    for split in ['train', 'val', 'test']:
        shutil.copytree(source_root / split, DATA_DRIVE_DIR / split)

    print('Dataset copié vers Drive :', DATA_DRIVE_DIR)

In [ ]:
# Copie le projet vers le runtime local Colab pour accélérer l'exécution.
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)

def _ignore(dir_path, names):
    ignored = {'venv', '__pycache__', '.ipynb_checkpoints', 'runs', 'checkpoints', 'logs', 'outputs'}
    if not COPY_DATA_TO_RUNTIME:
        ignored.add('data')
    return [name for name in names if name in ignored]

shutil.copytree(PROJECT_SRC, RUNTIME_DIR, ignore=_ignore)
os.chdir(RUNTIME_DIR)

TRAIN_DATA_DIR = RUNTIME_DIR / 'data' if COPY_DATA_TO_RUNTIME else DATA_DRIVE_DIR
print('RUNTIME_DIR    =', RUNTIME_DIR)
print('TRAIN_DATA_DIR =', TRAIN_DATA_DIR)
print('RUNS_ROOT      =', RUNS_ROOT)

## Étape 1 — Test Rapide

Commencez par un run de validation :
- `RUN_NAME = 'smoke_v1'`
- `EPOCHS = 1`

Cela permet de vérifier :
- que l'entraînement démarre,
- que `best.pth` et `last.pth` sont bien sauvegardés,
- que les métriques s'écrivent,
- que le Grad-CAM fonctionne.

In [ ]:
EPOCHS = 1
BATCH_SIZE = 2
LR = 1e-4
FREEZE_CNN = 0
VAL_STRATEGY = 'auto'
VAL_SPLIT = 0.10
NUM_WORKERS = 0
USE_PRETRAINED = False
USE_AMP = True
LOG_EVERY_STEPS = 10
MAX_TRAIN_BATCHES = 20
MAX_VAL_BATCHES = 10
EVALUATE_TEST = False

run_cmd = [
    sys.executable, '-u', 'run_experiment.py',
    '--run_name', RUN_NAME,
    '--runs_root', str(RUNS_ROOT),
    '--data_dir', str(TRAIN_DATA_DIR),
    '--epochs', str(EPOCHS),
    '--batch_size', str(BATCH_SIZE),
    '--lr', str(LR),
    '--freeze_cnn', str(FREEZE_CNN),
    '--val_strategy', VAL_STRATEGY,
    '--val_split', str(VAL_SPLIT),
    '--num_workers', str(NUM_WORKERS),
    '--log_every_steps', str(LOG_EVERY_STEPS),
    '--max_train_batches', str(MAX_TRAIN_BATCHES),
    '--max_val_batches', str(MAX_VAL_BATCHES),
]

if USE_PRETRAINED:
    run_cmd.append('--pretrained')
else:
    run_cmd.append('--no-pretrained')

if USE_AMP:
    run_cmd.append('--amp')

if EVALUATE_TEST:
    run_cmd.append('--evaluate_test')

print(' '.join(str(x) for x in run_cmd))


In [ ]:
import os
import subprocess

env = os.environ.copy()
env['PYTHONUNBUFFERED'] = '1'

process = subprocess.Popen(
    run_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env,
)

try:
    for line in process.stdout:
        print(line, end='')
finally:
    return_code = process.wait()

print('Return code:', return_code)


## Vérifier Le Run Et Les Métriques

Après le test ou l'entraînement sérieux, cette cellule affiche :
- l'état du run,
- le device utilisé,
- la durée,
- le nombre d'époques complétées,
- les dernières lignes de `metrics.csv`.

In [ ]:
import json
import pandas as pd
from IPython.display import display

RUN_DIR = RUNS_ROOT / RUN_NAME
STATUS_PATH = RUN_DIR / 'run_status.json'
METRICS_PATH = RUN_DIR / 'logs' / 'metrics.csv'
LOG_DIR = RUN_DIR / 'logs'
LOG_FILES = sorted(LOG_DIR.glob('train_*.log')) if LOG_DIR.exists() else []

if STATUS_PATH.exists():
    status = json.loads(STATUS_PATH.read_text())
    print(json.dumps(status, indent=2, ensure_ascii=False))
else:
    print('run_status.json introuvable:', STATUS_PATH)

if METRICS_PATH.exists():
    metrics_df = pd.read_csv(METRICS_PATH)
    display(metrics_df.tail())
else:
    print('metrics.csv introuvable:', METRICS_PATH)

if LOG_FILES:
    last_log = LOG_FILES[-1]
    print()
    print('===== Dernières lignes du log =====')
    print(last_log)
    print(''.join(last_log.read_text().splitlines(True)[-80:]))
else:
    print('Aucun log train_*.log trouvé dans', LOG_DIR)


## Tester Le Meilleur Modèle Et Voir Le Grad-CAM

Cette cellule prend `best.pth`, choisit une image de test, génère une visualisation Grad-CAM,
et sauvegarde l'image dans `outputs/sample_prediction.png`.

Vous pouvez modifier `SAMPLE_IMAGE` si vous voulez cibler un cas précis.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

RUN_DIR = RUNS_ROOT / RUN_NAME
BEST_CKPT = RUN_DIR / 'checkpoints' / 'best.pth'
assert BEST_CKPT.exists(), f'best.pth introuvable: {BEST_CKPT}'

sample_candidates = []
for split_class in [
    Path(TRAIN_DATA_DIR) / 'test' / 'PNEUMONIA',
    Path(TRAIN_DATA_DIR) / 'test' / 'NORMAL',
]:
    if split_class.exists():
        sample_candidates.extend(sorted(split_class.glob('*')))

SAMPLE_IMAGE = sample_candidates[0] if sample_candidates else None
OUTPUT_IMAGE = RUN_DIR / 'outputs' / 'sample_prediction.png'

if SAMPLE_IMAGE is None:
    print('Aucune image trouvée dans le dossier test/.')
else:
    predict_cmd = [
        sys.executable, 'predict.py',
        '--checkpoint', str(BEST_CKPT),
        '--image', str(SAMPLE_IMAGE),
        '--mc_passes', '10',
        '--save_vis', str(OUTPUT_IMAGE),
    ]
    print('Running:', ' '.join(str(x) for x in predict_cmd))
    subprocess.run(predict_cmd, check=True)
    display(Image(filename=str(OUTPUT_IMAGE)))

## Étape 2 — Entraînement Sérieux

Quand le test rapide est validé :
- soit vous gardez le même `RUN_NAME` et vous augmentez `EPOCHS`,
- soit vous créez un nouveau run, par exemple `RUN_NAME = 'main_v1'`.

Recommandation :
- test rapide : `RUN_NAME = 'smoke_v1'`, `EPOCHS = 1`
- vrai run : `RUN_NAME = 'main_v1'`, `EPOCHS = 10`

Rappel important :
- `EPOCHS` = nombre total d'époques visé pour le run.
- Exemple : si vous avez déjà fait `EPOCHS = 10`, mettez `EPOCHS = 20` pour continuer jusqu'à l'époque 20.

## Où sont vos fichiers ?

Tout est sauvegardé sur Drive ici :

```text
MyDrive/DPR_PFA4IADO/colab_runs/<RUN_NAME>/
├── checkpoints/
│   ├── best.pth
│   └── last.pth
├── logs/
│   ├── metrics.csv
│   └── train_*.log
├── outputs/
├── run_config.json
├── run_status.json
└── run_history.jsonl
```

Pour reprendre plus tard :
- rouvrez ce notebook,
- gardez le même `RUN_NAME`,
- augmentez `EPOCHS`,
- relancez seulement les cellules à partir des paramètres du run.